# 4. JsonOutputParser

The parser that turns the model's reply into a Python **dict** — and can parse **partial JSON while it
streams**. Reach for it when you want a dict (not a typed object) or you want to show output building
up live.

---

## 1. Simple Definition

> **Kid version:** Like the checklist parser, but it hands you a **plain labeled box** (a dictionary),
> and it's special: it can read the answer **while the AI is still typing**, showing pieces as they
> arrive.

**Professional definition:** `JsonOutputParser` parses model output into a `dict`/`list`, tolerating
code fences and minor noise, and supports **streaming partial JSON**. It can optionally take a Pydantic
schema just to generate format instructions.

```python
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
parser.parse('{"name": "Alice", "age": 30}')   # {'name': 'Alice', 'age': 30}
```

---

## 2. Why Does It Exist?

**The problem:** Raw `json.loads` is brittle — it fails if the model wraps JSON in ```json fences or
adds "Here you go:". And `PydanticOutputParser` can't stream partial results for live UIs.

### Before

```python
raw = model.invoke(prompt).content
data = json.loads(raw)     # 💥 fails on ```json fences or extra prose
```

### After

```python
parser = JsonOutputParser()
data = parser.parse(raw)   # robust: strips fences/noise → dict
# and in streaming, yields a growing partial dict as tokens arrive
```

A more forgiving `json.loads` that also streams and plugs into chains.

---

## 3. Real-Life Analogy

**A live sports scoreboard** ⚽. You don't wait for the final whistle — the score updates as goals
happen. Streaming `JsonOutputParser` is that scoreboard: `{"name":"Al"}` → `{"name":"Alice"}` →
`{"name":"Alice","age":30}` as the model generates.

---

## 4. Where It Fits in LangChain Architecture

```
BaseOutputParser
    │
    ▼
BaseCumulativeTransformOutputParser   ← re-parses the growing buffer for streaming
    │
    ▼
JsonOutputParser        → dict / list  (streams partials)
```

- Descends from a **cumulative transform** parser, which is exactly what enables partial-JSON
  streaming.
- Compared to its siblings:

| | `JsonOutputParser` | `PydanticOutputParser` | `StructuredOutputParser` |
|---|---|---|---|
| Returns | dict/list | validated object | dict of strings |
| Validation | none/light | full | none |
| Streaming | ✅ | ❌ | ❌ |
| Nesting | ✅ | ✅ | limited |

---

## 5. Internal Working

```
  MODEL TEXT (maybe ```json fenced, maybe with prose)
        │
        ▼
  CLEAN: strip code fences / surrounding text
        │
        ▼
  tolerant JSON parse → dict / list
        │
        ▼
  (streaming) each new token → re-parse the PARTIAL buffer → yield best-effort dict so far
```

---

## 6. Attributes / Methods

### `pydantic_object` *(optional)*

**Definition:** A Pydantic model used **only** to generate format instructions (output is still a
dict).

**Why it exists:** Describe the desired keys/types to the model while keeping dict output.

**When developers use it:** When you want strong instructions but prefer a dict.

```python
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="full name")
    age: int = Field(description="age in years")

parser = JsonOutputParser(pydantic_object=Person)
parser.get_format_instructions()   # instructions derived from Person
```

---

### `get_format_instructions()`

**Definition:** To generate instructions that are given to the LLM telling it what format its output should follow so that JsonOutputParser can parse it correctly. get_format_instructions() is a method of an output parser that generates formatting instructions for the LLM.

The returned text contains instructions along the lines of:
Return a JSON object.

Do not return the answer in natural language.
Return valid JSON that can be parsed.
The exact text depends on the LangChain version and parser configuration.

**Why it exists:** Tell the LLM what format its response must follow, so the JsonOutputParser can correctly parse that response.

```python
parser.get_format_instructions()
```

---

### `parse()`

**Definition:** Converts JSON text into a dict/list (tolerant of fences and noise).

```python
parser.parse('```json\n{"ok": true}\n```')   # {'ok': True}
```

---

## Streaming example (the standout feature)

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="full name")
    age: int = Field(description="age in years")
    hobbies: list[str] = Field(description="hobbies")

parser = JsonOutputParser(pydantic_object=Person)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract info. {format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI(model="gpt-4o-mini") | parser

for partial in chain.stream({"text": "Alice is 30, likes chess and running."}):
    print(partial)
# {}
# {'name': 'Alice'}
# {'name': 'Alice', 'age': 30}
# {'name': 'Alice', 'age': 30, 'hobbies': ['chess']}
# {'name': 'Alice', 'age': 30, 'hobbies': ['chess', 'running']}
```

---

In [2]:
from langchain_core.output_parsers import JsonOutputParser
JsonOutputParser().parse('{"name": "John", "age": 30, "city": "New York"}')

{'name': 'John', 'age': 30, 'city': 'New York'}

In [3]:
# you can see this code has produces an error it's output is in markdown format, something like this:
# Invalid json output: **Extracted Information:**  
# - **Customer Name:** Sachin  
# - **Issue:** Package arrived damaged  
# - **Request:** Refund

# Note: JsonOutputParser does NOT make the LLM produce JSON. It only parses/validates whatever the LLM produces.

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

# Initialize parser
parser = JsonOutputParser()

extract_info_template = PromptTemplate(
template="""
Extract information from this customer message.

Customer message:
{message}
""",
input_variables=["message"]
)

print(extract_info_template.format(message= "Hi, I'm Sachin. My package arrived damaged and I want a refund."))


Extract information from this customer message.

Customer message:
Hi, I'm Sachin. My package arrived damaged and I want a refund.



In [4]:
chain = extract_info_template | llm | parser

llm_result = chain.invoke({"message": "Hi, I'm Sachin. My package arrived damaged and I want a refund."})

llm_result

OutputParserException: Invalid json output: **Extracted Information:**  
- **Customer Name:** Sachin  
- **Issue:** Package arrived damaged  
- **Request:** Refund
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

##### Code Explanation:

##### 1. Basic Flow

The LangChain pipeline looks like this:

```text
PromptTemplate
      ↓
   ChatOllama
      ↓
JsonOutputParser
      ↓
Python Dictionary
```

More specifically:

```text
Prompt
   ↓
LLM
   ↓
Generates output
   ↓
JsonOutputParser
   ↓
Checks and parses the output
```

---

##### 2. What `JsonOutputParser` Actually Does

One of the most important things to understand is:

> **`JsonOutputParser` does NOT generate JSON.**

Many beginners assume:

```python
parser = JsonOutputParser()
```

means:

> "Make the LLM return JSON."

That is **not** what it means.

Instead, it means:

> "Take the LLM's response and try to parse it as JSON."

Think of `JsonOutputParser` as a **parser/validator**.

Its job is:

```text
JSON string
     ↓
JsonOutputParser
     ↓
Python object
```

It is **not**:

```text
Normal LLM response
     ↓
JsonOutputParser
     ↓
JSON
```

The LLM must first generate output that is valid JSON.

---

##### 3. What Happened in Our Example?

Suppose we have:

```python
parser = JsonOutputParser()

chain = prompt | llm | parser
```

The pipeline is:

```text
User Input
    ↓
PromptTemplate
    ↓
ChatOllama
    ↓
JsonOutputParser
    ↓
Python Dictionary
```

Now suppose the customer sends:

```text
Hi, I'm Sachin. My package arrived damaged and I want a refund.
```

The Qwen3 model might generate:

```text
**Extracted Information:**

- **Customer Name:** Sachin
- **Issue:** Package arrived damaged
- **Request:** Refund
```

This is perfectly understandable to a human.

However, it is **not valid JSON**.

---

##### 4. What Does the Parser Do?

The parser receives exactly what the LLM generated:

```text
**Extracted Information:**

- **Customer Name:** Sachin
- **Issue:** Package arrived damaged
- **Request:** Refund
```

It then tries to interpret that output as JSON.

Conceptually, it is doing something similar to:

```python
import json

json.loads("""
**Extracted Information:**

- **Customer Name:** Sachin
- **Issue:** Package arrived damaged
- **Request:** Refund
""")
```

Python cannot parse this as JSON because the input is Markdown/text, not a JSON object.

Therefore, Python raises:

```text
JSONDecodeError
```

LangChain then reports:

```text
OutputParserException: Invalid json output
```

---

##### 5. Why Did We Get the Error?

The complete flow was:

```text
Customer Message
      ↓
"Hi, I'm Sachin. My package arrived damaged
 and I want a refund."
      ↓
PromptTemplate
      ↓
ChatOllama / Qwen3
      ↓
**Extracted Information:**
- Customer Name: Sachin
- Issue: Package arrived damaged
- Request: Refund
      ↓
JsonOutputParser
      ↓
❌ Not valid JSON
      ↓
JSONDecodeError
      ↓
OutputParserException
```

The problem is **not that the parser is broken**.

The parser is doing exactly what it is supposed to do.

The problem is that the **LLM did not return valid JSON**.

---

##### 6. The Key Difference

Remember this distinction:

```text
LLM
↓
GENERATES the response
```

while:

```text
JsonOutputParser
↓
PARSES the response
```

Therefore:

```text
LLM = Generator
JsonOutputParser = Parser
```

---

##### 7. What We Actually Want

We want the LLM to generate:

```json
{
    "customer_name": "Sachin",
    "issue": "Package arrived damaged",
    "requested_action": "Refund"
}
```

Then the pipeline becomes:

```text
Customer Message
      ↓
PromptTemplate
      ↓
Qwen3
      ↓
Valid JSON
      ↓
JsonOutputParser
      ↓
Python Dictionary
```

The final result can then be used in Python:

```python
result["customer_name"]
```

Output:

```text
Sachin
```

Or:

```python
result["issue"]
```

Output:

```text
Package arrived damaged
```

Or:

```python
result["requested_action"]
```

Output:

```text
Refund
```

---

##### 8. Important Mental Model

The most important thing to remember is:

```text
Prompt tells
     ↓
LLM generates
     ↓
Parser parses
```

Or simply:

```text
LLM = Generate
Parser = Parse
```

`JsonOutputParser` does not magically convert any LLM response into JSON.

The LLM needs to produce JSON first, and then `JsonOutputParser` can parse it into a Python object.


In [5]:
# this is the same code as above but with the prompt updated to ask for JSON output, which will work with the JsonOutputParser

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

parser = JsonOutputParser()

extract_info_template = PromptTemplate(
template="""
Extract information from this customer message.

Customer message:
{message}

Return the result as JSON with these fields:
- customer_name
- problem
- requested_action
""",
input_variables=["message"]
)

print(extract_info_template.format(message= "Hi, I'm Sachin. My package arrived damaged and I want a refund."))

chain = extract_info_template | llm | parser

llm_result = chain.invoke({"message": "Hi, I'm Sachin. My package arrived damaged and I want a refund."})

llm_result


Extract information from this customer message.

Customer message:
Hi, I'm Sachin. My package arrived damaged and I want a refund.

Return the result as JSON with these fields:
- customer_name
- problem
- requested_action



{'customer_name': 'Sachin',
 'problem': 'package arrived damaged',
 'requested_action': 'refund'}

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

parser = JsonOutputParser()

topic_template = PromptTemplate(template='Give me 5 facts about {topic} \n{format_instruction}',
                                input_variables=['topic'],
                                partial_variables={'format_instruction': parser.get_format_instructions()}
                               )

prompt = topic_template.format(topic='black hole')

print(prompt)

Give me 5 facts about black hole 
Return a JSON object.


In [7]:
chain = topic_template | llm | parser

llm_result = chain.invoke({'topic':'black hole'})

llm_result

{'facts': [{'number': 1,
   'fact': 'A black hole is a region in space where gravity is so intense that not even light can escape its pull.'},
  {'number': 2,
   'fact': 'Black holes form when massive stars collapse under their own gravity, creating a point of infinite density called a singularity.'},
  {'number': 3,
   'fact': 'The event horizon is the boundary around a black hole beyond which nothing, including light, can escape its gravitational grasp.'},
  {'number': 4,
   'fact': 'Supermassive black holes, millions to billions times more massive than the Sun, reside at the centers of most galaxies, including our Milky Way.'},
  {'number': 5,
   'fact': 'Time dilation near a black hole causes time to slow down significantly for an observer close to it compared to someone farther away.'}]}

In [8]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

parser = JsonOutputParser()

topic_template = PromptTemplate(template='Give me the name, age, and city of a fictional person \n{format_instruction}',
                                partial_variables={'format_instruction': parser.get_format_instructions()}
                               )

prompt = topic_template.format()

print(prompt)

Give me the name, age, and city of a fictional person 
Return a JSON object.


In [9]:
chain = topic_template | llm | parser

llm_result = chain.invoke({})

llm_result

{'name': 'Elena Voss', 'age': 28, 'city': 'Lumina City'}

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="full name")
    age: int = Field(description="age in years")
    hobbies: list[str] = Field(description="hobbies")

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

parser = JsonOutputParser(pydantic_object=Person)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract info. {format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

for partial in chain.stream({"text": "Alice is 30, likes chess and running."}):
    print(partial)

{}
{'name': ''}
{'name': 'Alice'}
{'name': 'Alice', 'age': 3}
{'name': 'Alice', 'age': 30}
{'name': 'Alice', 'age': 30, 'hobbies': ['']}
{'name': 'Alice', 'age': 30, 'hobbies': ['ch']}
{'name': 'Alice', 'age': 30, 'hobbies': ['chess']}
{'name': 'Alice', 'age': 30, 'hobbies': ['chess', '']}
{'name': 'Alice', 'age': 30, 'hobbies': ['chess', 'running']}
